# Zero-day novelty detection: reservoir and detector scaled together

Same normal-only protocol as `20_zeroday_novelty`; the detector now grows with
the reservoir: each ladder step pairs a bigger reservoir with a wider
autoencoder (config: `configs/novelty_ladder.yaml`). The original linear
detector (`pca32`) runs at every step as the baseline, and IsolationForest on
the raw features is the classical reference.

In [ ]:
# Setup: make the repo importable from notebooks/ (or from the repo root).
import sys
from pathlib import Path

REPO = Path.cwd().resolve()
REPO = REPO.parent if REPO.name == "notebooks" else REPO
sys.path.insert(0, str(REPO))

import pandas as pd
from elements import config
from elements.eval.results import load_results
from elements.experiments.runner import plan, run
from elements.viz import scaling
pd.set_option("display.precision", 4)
pd.set_option("display.width", 160)
from elements.viz import novelty as nv

In [ ]:
CFG = config.load(REPO / "configs" / "novelty_ladder.yaml")
FIGS = CFG.out_dir / "figs"
print(plan(CFG))

### Run

Off by default so opening the notebook never starts a long run. Set `RUN = True`
(or run `python -m elements.run configs/novelty_ladder.yaml` from the repo root). Every trained head is
appended to `results.jsonl` as it finishes, so an interrupted run resumes where
it stopped.

In [ ]:
RUN = False
if RUN:
    run(CFG)

## 1. The ladder

In [ ]:
DF = load_results(CFG.out_dir, keep=("per_type",))
scaling.ladder(DF, "macro_auc", out=FIGS / "ladder_macro_auc.png");
scaling.ladder(DF, "det@0.01", out=FIGS / "ladder_det1.png");

In [ ]:
scaling.ladder_table(DF, "macro_auc")

In [ ]:
scaling.ladder_table(DF, "det@0.01")

## 2. Per attack, along the ladder

In [ ]:
nv.per_attack_heatmap(DF, "quantum", "ladder", out=FIGS / "per_attack_quantum.png");

In [ ]:
nv.per_attack_heatmap(DF, "quantum", "ladder", value="det@0.01",
                      out=FIGS / "per_attack_quantum_det1.png");

## 3. Everything, sorted

In [ ]:
nv.detection_table(DF)

In [ ]:
scaling.best_table(DF, "macro_auc")